In [24]:
import os
import glob
import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
from tqdm.auto import tqdm

/Users/soeren.becker/micromamba/envs/env_pertpy/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [34]:
def combine_number_of_cell_per_donor(adata1, adata2):
    shared_idcs = adata1.obs.index.intersection(adata2.obs.index)

    unique_idcs1 = adata1.obs.index[~np.isin(adata1.obs.index, adata2.obs.index)]
    unique_idcs2 = adata2.obs.index[~np.isin(adata2.obs.index, adata1.obs.index)]
    
    adata1_shared = adata1[shared_idcs]
    adata2_shared = adata2[shared_idcs]
    
    combined = (adata1_shared.obs.psbulk_n_cells + adata2_shared.obs.psbulk_n_cells).to_frame()
    new_idcs = [*shared_idcs]
    for _idx1 in unique_idcs1:
        combined.loc[_idx1, "psbulk_n_cells"] = adata1.obs.loc[_idx1].psbulk_n_cells
        new_idcs.append(_idx1)
    
    for _idx2 in unique_idcs2:
        combined.loc[_idx2, "psbulk_n_cells"] = adata2.obs.loc[_idx2].psbulk_n_cells
        new_idcs.append(_idx2)

    return combined, new_idcs

def combine_counts(adata1, adata2):
    assert np.all(adata1.var_names == adata2.var_names)
    shared_idcs = adata1.obs.index.intersection(adata2.obs.index)
    unique_idcs1 = adata1.obs.index[~np.isin(adata1.obs.index, adata2.obs.index)]
    unique_idcs2 = adata2.obs.index[~np.isin(adata2.obs.index, adata1.obs.index)]
    adata1_shared = adata1[shared_idcs]
    adata2_shared = adata2[shared_idcs]
    assert np.all(adata1_shared.obs.index == adata2_shared.obs.index)
    
    X = adata1_shared.X + adata2_shared.X
    new_idcs = [*shared_idcs]
    if len(unique_idcs1) > 0:
        X = np.vstack([X, adata1[unique_idcs1].X])
        new_idcs.extend(unique_idcs1)
    if len(unique_idcs2) > 0:
        X = np.vstack([X, adata2[unique_idcs2].X])
        new_idcs.extend(unique_idcs2)
        
    return X, new_idcs

def combine_frac_expression_cells_total(adata1, adata2):
    assert all(adata1.var.index == adata2.var.index)
    n1 = adata1.obs.psbulk_n_cells.sum()
    n2 = adata2.obs.psbulk_n_cells.sum()
    f1 = adata1.var.frac_expressing_cells_total
    f2 = adata2.var.frac_expressing_cells_total
    m1 = n1 * f1
    m2 = n2 * f2
    return ((m1+m2) / (n1+n2)).to_frame()

def combine_pseudobulks(adata1, adata2, new_celltype_label):
    X, new_idcs_counts = combine_counts(adata1, adata2)
    num_cells, new_idcs_num_cells = combine_number_of_cell_per_donor(adata1, adata2)
    assert new_idcs_counts == new_idcs_num_cells
    frac_expressing_cells = combine_frac_expression_cells_total(adata1, adata2)

    obs = num_cells
    obs.loc[:, "celltype"] = new_celltype_label
    
    shared_idcs = adata1.obs.index.intersection(adata2.obs.index)
    unique_idcs1 = adata1.obs.index[~np.isin(adata1.obs.index, adata2.obs.index)]
    unique_idcs2 = adata2.obs.index[~np.isin(adata2.obs.index, adata1.obs.index)]

    
    donors = adata1[shared_idcs].obs.donor
    for _idx1 in unique_idcs1:
        donors.loc[_idx1] = adata1.obs.loc[_idx1].donor
    for _idx2 in unique_idcs2:
        donors.loc[_idx2] = adata2.obs.loc[_idx2].donor
    obs.loc[:, "donor"] = donors

    
    bc1_well = adata1[shared_idcs].obs.bc1_well
    for _idx1 in unique_idcs1:
        bc1_well.loc[_idx1] = adata1.obs.loc[_idx1].bc1_well
    for _idx2 in unique_idcs2:
        bc1_well.loc[_idx2] = adata2.obs.loc[_idx2].bc1_well
    obs.loc[:, "bc1_well"] = bc1_well

    cytokine1 = adata1.obs.cytokine.unique()
    cytokine2 = adata2.obs.cytokine.unique()
    
    assert len(cytokine1) == 1
    assert len(cytokine2) == 1
    assert cytokine1[0] == cytokine2[0]
    obs.loc[:, "cytokine"] = cytokine1[0]

    adata_sum = ad.AnnData(
        X = X,
        obs=obs,
        var=frac_expressing_cells,
    )
    return adata_sum

In [17]:
pdata_pbs1.obs.index == pdata_pbs2.obs.index

array([ True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True])

### Combine CD14 and CD16 monocytes

In [35]:
pdata_pbs1 = sc.read_h5ad(f"/Users/soeren.becker/repos/HumanCytokineDict/data/pseudobulks_unfiltered_pbs_by_well/pbulk_CD14_Mono_PBS_sum.h5ad")
pdata_pbs2 = sc.read_h5ad(f"/Users/soeren.becker/repos/HumanCytokineDict/data/pseudobulks_unfiltered_pbs_by_well/pbulk_CD16_Mono_PBS_sum.h5ad")

In [36]:
adata_sum = combine_pseudobulks(pdata_pbs1, pdata_pbs2, "Mono")

In [37]:
adata_sum.obs

,psbulk_n_cells,celltype,donor,bc1_well,cytokine
Donor1_H10,2476.0,Mono,Donor1,H10,PBS
Donor10_H10,2238.0,Mono,Donor10,H10,PBS
Donor11_H10,2685.0,Mono,Donor11,H10,PBS
Donor12_H10,2101.0,Mono,Donor12,H10,PBS
Donor2_H10,1988.0,Mono,Donor2,H10,PBS
...,...,...,...,...,...
Donor5_H9,630.0,Mono,Donor5,H9,PBS
Donor6_H9,1167.0,Mono,Donor6,H9,PBS
Donor7_H9,350.0,Mono,Donor7,H9,PBS
Donor8_H9,1382.0,Mono,Donor8,H9,PBS


In [38]:
for idx in tqdm(adata_sum.obs.index):

    n1 = pdata_pbs1.obs.loc[idx].psbulk_n_cells + pdata_pbs2.obs.loc[idx].psbulk_n_cells
    n2 = adata_sum.obs.loc[idx].psbulk_n_cells
    assert n1 == n2

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 72/72 [00:00<00:00, 2565.21it/s]


In [39]:
adata_sum.write_h5ad("/Users/soeren.becker/repos/HumanCytokineDict/data/pseudobulks_unfiltered_pbs_by_well/pbulk_Mono_PBS_sum.h5ad")

### Combined NK dim and NK bright cells

In [41]:
pdata_pbs1 = sc.read_h5ad(f"/Users/soeren.becker/repos/HumanCytokineDict/data/pseudobulks_unfiltered_pbs_by_well/pbulk_CD56-dim_NK_PBS_sum.h5ad")
pdata_pbs2 = sc.read_h5ad(f"/Users/soeren.becker/repos/HumanCytokineDict/data/pseudobulks_unfiltered_pbs_by_well/pbulk_CD56-bright_NK_PBS_sum.h5ad")

In [44]:
adata_sum = combine_pseudobulks(pdata_pbs1, pdata_pbs2, "NK")

In [46]:
adata_sum.obs

,psbulk_n_cells,celltype,donor,bc1_well,cytokine
Donor1_H10,805.0,NK,Donor1,H10,PBS
Donor10_H10,442.0,NK,Donor10,H10,PBS
Donor11_H10,540.0,NK,Donor11,H10,PBS
Donor12_H10,547.0,NK,Donor12,H10,PBS
Donor2_H10,584.0,NK,Donor2,H10,PBS
...,...,...,...,...,...
Donor5_H9,704.0,NK,Donor5,H9,PBS
Donor6_H9,922.0,NK,Donor6,H9,PBS
Donor7_H9,236.0,NK,Donor7,H9,PBS
Donor8_H9,489.0,NK,Donor8,H9,PBS


In [47]:
adata_sum.write_h5ad("/Users/soeren.becker/repos/HumanCytokineDict/data/pseudobulks_unfiltered_pbs_by_well/pbulk_NK_PBS_sum.h5ad")

In [48]:
sc.read_h5ad("/Users/soeren.becker/repos/HumanCytokineDict/data/pseudobulks_unfiltered_pbs_by_well/pbulk_NK_PBS_sum.h5ad")

AnnData object with n_obs × n_vars = 72 × 40352
    obs: 'psbulk_n_cells', 'celltype', 'donor', 'bc1_well', 'cytokine'
    var: 'frac_expressing_cells_total'